In [1]:
# ============================================================
# BiasTrace COMPAS RCA MI Ablation
# Compare label-agnostic MI I(Z;A) vs label-conditional MI I(Z;A|Y)
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr


# ------------------------------------------------------------
# 0. Config
# ------------------------------------------------------------

COMPAS_CSV_PATH = "/content/drive/MyDrive/Paper_Bias/LTDD/LTDD-main/datasets/compas-scores-two-years.csv"  # 수정

RANDOM_SEEDS = [0, 1, 2, 3, 4]
TOPK_VALUES = [1, 3, 5, 10]

PROTECTED_POSITIVE_RACE = "African-American"
PROTECTED_REFERENCE_RACE = "Caucasian"

TARGET_COL = "two_year_recid"
A_COL = "race"

# y=1을 favorable outcome으로 둔다.
# COMPAS에서는 two_year_recid == 0 이 "재범하지 않음"이므로 favorable.
FAVORABLE_Y_VALUE = 1


# ------------------------------------------------------------
# 1. COMPAS loading and preprocessing
# ------------------------------------------------------------

def load_compas_raw(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"COMPAS CSV not found: {path}\n"
            "Please upload compas-scores-two-years.csv to Colab or update COMPAS_CSV_PATH."
        )

    df = pd.read_csv(path)

    # Common COMPAS filtering convention
    # Keep only African-American and Caucasian groups.
    df = df[df[A_COL].isin([PROTECTED_POSITIVE_RACE, PROTECTED_REFERENCE_RACE])].copy()

    if "days_b_screening_arrest" in df.columns:
        df = df[
            (df["days_b_screening_arrest"] >= -30)
            & (df["days_b_screening_arrest"] <= 30)
        ].copy()

    if "is_recid" in df.columns:
        df = df[df["is_recid"] != -1].copy()

    if "c_charge_degree" in df.columns:
        df = df[df["c_charge_degree"] != "O"].copy()

    if "score_text" in df.columns:
        df = df[df["score_text"] != "N/A"].copy()

    df = df.reset_index(drop=True)
    return df


def make_compas_base(df: pd.DataFrame):
    """
    Returns:
      base_df: raw feature dataframe for fault injection
      A: protected attribute, A=1 for African-American, A=0 for Caucasian
      y: favorable label, y=1 for no recidivism, y=0 for recidivism
    """
    A = (df[A_COL] == PROTECTED_POSITIVE_RACE).astype(int).to_numpy()
    y = (df[TARGET_COL] == 0).astype(int).to_numpy()

    drop_cols = [
        # target / protected
        TARGET_COL, A_COL,

        # identifiers / names
        "id", "name", "first", "last",

        # dates
        "compas_screening_date", "dob", "c_jail_in", "c_jail_out",
        "c_offense_date", "c_arrest_date", "r_offense_date",
        "r_jail_in", "r_jail_out", "vr_offense_date",

        # leakage / COMPAS output columns
        "decile_score", "score_text", "v_decile_score", "v_score_text",
        "is_recid", "is_violent_recid",

        # case identifiers / post-outcome variables
        "c_case_number", "r_case_number", "vr_case_number",
        "r_charge_degree", "r_charge_desc",
        "vr_charge_degree", "vr_charge_desc",
        "violent_recid"
    ]

    keep_cols = [c for c in df.columns if c not in drop_cols]
    base_df = df[keep_cols].copy()

    # Keep only useful columns
    # Remove columns that are entirely missing or too high-cardinality free text.
    remove_more = []
    for c in base_df.columns:
        if base_df[c].isna().all():
            remove_more.append(c)
        elif base_df[c].dtype == "object" and base_df[c].nunique(dropna=True) > 50:
            remove_more.append(c)

    base_df = base_df.drop(columns=remove_more, errors="ignore")

    return base_df, A, y


# ------------------------------------------------------------
# 2. Fault injection scenarios for RCA MI ablation
# ------------------------------------------------------------

def inject_missingness(df, A, y, rng, feature="days_b_screening_arrest", rate=0.30):
    fdf = df.copy()
    mask = (A == 1) & (rng.random(len(fdf)) < rate)
    if feature in fdf.columns:
        fdf.loc[mask, feature] = np.nan
    return fdf, y.copy(), {"missing::" + feature, feature}


def inject_scale(df, A, y, rng, feature="priors_count", factor=1.50):
    fdf = df.copy()
    if feature in fdf.columns:
        fdf.loc[A == 1, feature] = fdf.loc[A == 1, feature].astype(float) * factor
    return fdf, y.copy(), {feature}


def inject_shift(df, A, y, rng, feature="priors_count", shift=2.0):
    fdf = df.copy()
    if feature in fdf.columns:
        fdf.loc[A == 1, feature] = fdf.loc[A == 1, feature].astype(float) + shift
    return fdf, y.copy(), {feature}


def inject_clip(df, A, y, rng, feature="priors_count", upper_quantile=0.70):
    fdf = df.copy()
    if feature in fdf.columns:
        upper = fdf[feature].quantile(upper_quantile)
        idx = (A == 1)
        fdf.loc[idx, feature] = np.minimum(fdf.loc[idx, feature].astype(float), upper)
    return fdf, y.copy(), {feature}


def inject_conditional_selection(df, A, y, rng, feature="priors_count", threshold=2, rate=0.35):
    """
    Remove a fraction of A=1 samples with priors_count > threshold.
    Returns filtered df, A, y must also be filtered later.
    """
    fdf = df.copy()
    cond = (A == 1)
    if feature in fdf.columns:
        cond = cond & (fdf[feature].to_numpy() > threshold)

    remove = cond & (rng.random(len(fdf)) < rate)
    keep = ~remove

    return fdf.loc[keep].reset_index(drop=True), A[keep], y[keep], {feature}


def inject_missing_plus_scale(df, A, y, rng):
    fdf, y2, carriers1 = inject_missingness(
        df, A, y, rng, feature="days_b_screening_arrest", rate=0.20
    )
    fdf, y2, carriers2 = inject_scale(
        fdf, A, y2, rng, feature="priors_count", factor=1.25
    )
    return fdf, y2, carriers1.union(carriers2)


def inject_scale_plus_masking(df, A, y, rng):
    # RCA 관점에서는 upstream feature carrier만 본다.
    # downstream threshold carrier는 이 MI feature ablation에서 제외.
    fdf, y2, carriers = inject_scale(
        df, A, y, rng, feature="priors_count", factor=1.50
    )
    return fdf, y2, carriers


SCENARIOS = {
    "S1_CONDITIONAL_SELECTION_PRIORS": "conditional_selection",
    "S2_MISSING_SCREENING_DAYS": "missingness",
    "S3_SCALE_PRIORS": "scale",
    "S3_SHIFT_PRIORS": "shift",
    "S3_CLIP_PRIORS": "clip",
    "M2_MISSING_PLUS_SCALE": "missing_plus_scale",
    "M3_SCALE_PLUS_MASKING": "scale_plus_masking",
}


def apply_scenario(base_df, A, y, scenario_name, seed):
    rng = np.random.default_rng(seed)
    kind = SCENARIOS[scenario_name]

    if kind == "conditional_selection":
        return inject_conditional_selection(base_df, A, y, rng)

    if kind == "missingness":
        fdf, y2, carriers = inject_missingness(base_df, A, y, rng)
        return fdf, A.copy(), y2, carriers

    if kind == "scale":
        fdf, y2, carriers = inject_scale(base_df, A, y, rng)
        return fdf, A.copy(), y2, carriers

    if kind == "shift":
        fdf, y2, carriers = inject_shift(base_df, A, y, rng)
        return fdf, A.copy(), y2, carriers

    if kind == "clip":
        fdf, y2, carriers = inject_clip(base_df, A, y, rng)
        return fdf, A.copy(), y2, carriers

    if kind == "missing_plus_scale":
        fdf, y2, carriers = inject_missing_plus_scale(base_df, A, y, rng)
        return fdf, A.copy(), y2, carriers

    if kind == "scale_plus_masking":
        fdf, y2, carriers = inject_scale_plus_masking(base_df, A, y, rng)
        return fdf, A.copy(), y2, carriers

    raise ValueError(f"Unknown scenario kind: {kind}")


# ------------------------------------------------------------
# 3. Carrier matrix construction
# ------------------------------------------------------------

def build_carrier_matrix(raw_df: pd.DataFrame):
    """
    Construct RCA carrier matrix.
    Includes:
      - numeric features
      - one-hot categorical features
      - missingness indicators for columns with missing values
    Returns:
      X: numpy array
      feature_names: list[str]
      discrete_mask: boolean array indicating discrete features
    """
    df = raw_df.copy()

    carrier_parts = []
    feature_names = []
    discrete_mask = []

    # Missingness indicators before imputation
    for c in df.columns:
        if df[c].isna().any():
            miss = df[c].isna().astype(int).to_numpy().reshape(-1, 1)
            carrier_parts.append(miss)
            feature_names.append(f"missing::{c}")
            discrete_mask.append(True)

    # Separate numeric and categorical
    numeric_cols = df.select_dtypes(include=[np.number, "bool"]).columns.tolist()
    categorical_cols = [c for c in df.columns if c not in numeric_cols]

    # Numeric imputation
    if numeric_cols:
        num_df = df[numeric_cols].copy()
        for c in numeric_cols:
            med = num_df[c].median()
            if pd.isna(med):
                med = 0.0
            num_df[c] = num_df[c].fillna(med)

        # Keep numeric values as-is. Standardization is optional for MI,
        # but helps numerical stability for continuous MI estimator.
        scaler = StandardScaler()
        X_num = scaler.fit_transform(num_df.to_numpy(dtype=float))

        carrier_parts.append(X_num)
        feature_names.extend(numeric_cols)
        discrete_mask.extend([False] * len(numeric_cols))

    # Categorical imputation + one-hot
    if categorical_cols:
        cat_df = df[categorical_cols].copy()
        for c in categorical_cols:
            mode = cat_df[c].mode(dropna=True)
            fill_value = mode.iloc[0] if len(mode) > 0 else "MISSING"
            cat_df[c] = cat_df[c].fillna(fill_value).astype(str)

        dummies = pd.get_dummies(cat_df, drop_first=False)
        X_cat = dummies.to_numpy(dtype=float)

        carrier_parts.append(X_cat)
        feature_names.extend(dummies.columns.tolist())
        discrete_mask.extend([True] * X_cat.shape[1])

    if not carrier_parts:
        raise ValueError("No usable carrier features found.")

    X = np.hstack(carrier_parts)
    discrete_mask = np.array(discrete_mask, dtype=bool)

    return X, feature_names, discrete_mask


# ------------------------------------------------------------
# 4. MI and conditional MI scoring
# ------------------------------------------------------------

def safe_mutual_info(X, A, discrete_mask, seed=0):
    """
    Wrapper around sklearn mutual_info_classif with safeguards.
    """
    A = np.asarray(A).astype(int)

    # If only one protected group exists, MI is undefined; return zeros.
    if len(np.unique(A)) < 2:
        return np.zeros(X.shape[1])

    # sklearn can fail if there are too few samples in rare subsets.
    try:
        scores = mutual_info_classif(
            X,
            A,
            discrete_features=discrete_mask,
            random_state=seed,
            n_neighbors=3
        )
    except Exception as e:
        print(f"[WARN] mutual_info_classif failed: {e}")
        scores = np.zeros(X.shape[1])

    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    return scores


def label_agnostic_mi(X, A, y, discrete_mask, seed=0):
    return safe_mutual_info(X, A, discrete_mask, seed=seed)


def label_conditional_mi(X, A, y, discrete_mask, seed=0):
    """
    Approximate I(Z;A|Y) = sum_y P(Y=y) I(Z;A | Y=y)
    using sklearn MI inside each label stratum.
    """
    y = np.asarray(y).astype(int)
    n = len(y)
    scores = np.zeros(X.shape[1], dtype=float)

    for yy in np.unique(y):
        idx = (y == yy)
        weight = idx.mean()

        # Need enough samples and both A classes.
        if idx.sum() < 20:
            continue
        if len(np.unique(np.asarray(A)[idx])) < 2:
            continue

        scores_y = safe_mutual_info(
            X[idx],
            np.asarray(A)[idx],
            discrete_mask,
            seed=seed
        )
        scores += weight * scores_y

    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    return scores


# ------------------------------------------------------------
# 5. Ranking and evaluation
# ------------------------------------------------------------

def normalize_feature_name(name: str) -> str:
    return str(name).replace(" ", "_").lower()


def is_carrier_match(feature_name: str, gt_carriers: set):
    """
    Match exact carrier or encoded one-hot/prefix carrier.
    For numeric feature priors_count, exact match works.
    For missing::days_b_screening_arrest, exact match works.
    """
    f = normalize_feature_name(feature_name)

    for gt in gt_carriers:
        g = normalize_feature_name(gt)

        if f == g:
            return True

        # For one-hot expanded names, allow prefix match.
        # Example: c_charge_degree_F should match c_charge_degree if needed.
        if f.startswith(g + "_"):
            return True

        # For missingness carrier
        if g.startswith("missing::") and f == g:
            return True

    return False


def rank_features(scores, feature_names):
    order = np.argsort(-scores)
    ranked = []
    for rank, idx in enumerate(order, start=1):
        ranked.append({
            "rank": rank,
            "feature": feature_names[idx],
            "score": float(scores[idx])
        })
    return ranked


def evaluate_ranked(ranked, gt_carriers, topk_values=TOPK_VALUES):
    ranks = [
        row["rank"]
        for row in ranked
        if is_carrier_match(row["feature"], gt_carriers)
    ]

    best_rank = min(ranks) if ranks else np.inf
    out = {"best_rank": best_rank if np.isfinite(best_rank) else None}

    for k in topk_values:
        out[f"hit@{k}"] = int(np.isfinite(best_rank) and best_rank <= k)

    return out


def spearman_between_scores(scores_a, scores_b):
    if np.all(scores_a == scores_a[0]) or np.all(scores_b == scores_b[0]):
        return np.nan
    corr, _ = spearmanr(scores_a, scores_b)
    return float(corr)


# ------------------------------------------------------------
# 6. Main experiment
# ------------------------------------------------------------

def run_compas_mi_ablation(
    compas_csv_path=COMPAS_CSV_PATH,
    scenarios=None,
    seeds=RANDOM_SEEDS,
    save_prefix="biastrace_compas_mi_ablation"
):
    if scenarios is None:
        scenarios = list(SCENARIOS.keys())

    raw = load_compas_raw(compas_csv_path)
    base_df, A, y = make_compas_base(raw)

    print("========== COMPAS DATA INFO ==========")
    print("Rows:", len(base_df))
    print("Protected A count [0,1]:", np.bincount(A))
    print("Favorable y count [0,1]:", np.bincount(y))
    print("Base feature columns:", len(base_df.columns))
    print(base_df.columns.tolist()[:30])

    rows = []
    top_rows = []

    for scenario in scenarios:
        for seed in seeds:
            fdf, Af, yf, gt_carriers = apply_scenario(base_df, A, y, scenario, seed)

            X, feature_names, discrete_mask = build_carrier_matrix(fdf)

            scores_mi = label_agnostic_mi(X, Af, yf, discrete_mask, seed=seed)
            scores_cmi = label_conditional_mi(X, Af, yf, discrete_mask, seed=seed)

            ranked_mi = rank_features(scores_mi, feature_names)
            ranked_cmi = rank_features(scores_cmi, feature_names)

            eval_mi = evaluate_ranked(ranked_mi, gt_carriers)
            eval_cmi = evaluate_ranked(ranked_cmi, gt_carriers)

            corr = spearman_between_scores(scores_mi, scores_cmi)

            for mi_type, ranked, ev in [
                ("I(Z;A)", ranked_mi, eval_mi),
                ("I(Z;A|Y)", ranked_cmi, eval_cmi),
            ]:
                row = {
                    "dataset": "COMPAS",
                    "scenario": scenario,
                    "seed": seed,
                    "mi_type": mi_type,
                    "gt_carriers": ";".join(sorted(gt_carriers)),
                    "best_rank": ev["best_rank"],
                    "rank_corr_mi_vs_cmi": corr,
                }
                for k in TOPK_VALUES:
                    row[f"hit@{k}"] = ev[f"hit@{k}"]
                rows.append(row)

                # Save top-10 features for inspection
                for r in ranked[:10]:
                    top_rows.append({
                        "dataset": "COMPAS",
                        "scenario": scenario,
                        "seed": seed,
                        "mi_type": mi_type,
                        "rank": r["rank"],
                        "feature": r["feature"],
                        "score": r["score"],
                        "gt_carriers": ";".join(sorted(gt_carriers)),
                        "is_gt_match": int(is_carrier_match(r["feature"], gt_carriers))
                    })

    result_df = pd.DataFrame(rows)
    top_df = pd.DataFrame(top_rows)

    summary = (
        result_df
        .groupby(["dataset", "mi_type"])
        .agg(
            n=("scenario", "count"),
            rca_r1=("hit@1", "mean"),
            rca_r3=("hit@3", "mean"),
            rca_r5=("hit@5", "mean"),
            rca_r10=("hit@10", "mean"),
            mean_best_rank=("best_rank", "mean"),
            mean_rank_corr=("rank_corr_mi_vs_cmi", "mean"),
        )
        .reset_index()
    )

    scenario_summary = (
        result_df
        .groupby(["scenario", "mi_type"])
        .agg(
            n=("seed", "count"),
            rca_r1=("hit@1", "mean"),
            rca_r3=("hit@3", "mean"),
            rca_r5=("hit@5", "mean"),
            mean_best_rank=("best_rank", "mean"),
            mean_rank_corr=("rank_corr_mi_vs_cmi", "mean"),
        )
        .reset_index()
    )

    # Convert to percentage for readability
    for df in [summary, scenario_summary]:
        for col in ["rca_r1", "rca_r3", "rca_r5", "rca_r10"]:
            if col in df.columns:
                df[col] = df[col] * 100.0

    result_path = f"{save_prefix}_per_run.csv"
    top_path = f"{save_prefix}_top10.csv"
    summary_path = f"{save_prefix}_summary.csv"
    scenario_summary_path = f"{save_prefix}_scenario_summary.csv"

    result_df.to_csv(result_path, index=False)
    top_df.to_csv(top_path, index=False)
    summary.to_csv(summary_path, index=False)
    scenario_summary.to_csv(scenario_summary_path, index=False)

    print("\n========== SUMMARY ==========")
    print(summary.to_string(index=False))

    print("\n========== SCENARIO SUMMARY ==========")
    print(scenario_summary.to_string(index=False))

    print("\nSaved files:")
    print(" -", result_path)
    print(" -", top_path)
    print(" -", summary_path)
    print(" -", scenario_summary_path)

    return result_df, top_df, summary, scenario_summary


# ------------------------------------------------------------
# 7. Run
# ------------------------------------------------------------

result_df, top_df, summary_df, scenario_summary_df = run_compas_mi_ablation()

========== COMPAS DATA INFO ==========
Rows: 5278
Protected A count [0,1]: [2103 3175]
Favorable y count [0,1]: [2483 2795]
Base feature columns: 16
['sex', 'age', 'age_cat', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'days_b_screening_arrest', 'c_days_from_compas', 'c_charge_degree', 'r_days_from_arrest', 'type_of_assessment', 'v_type_of_assessment', 'start', 'end', 'event']


/tmp/ipykernel_2341/1859564747.py:136: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.  6.  4.5 ... 0.  0.  4.5]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fdf.loc[A == 1, feature] = fdf.loc[A == 1, feature].astype(float) * factor
/tmp/ipykernel_2341/1859564747.py:136: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.  6.  4.5 ... 0.  0.  4.5]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  fdf.loc[A == 1, feature] = fdf.loc[A == 1, feature].astype(float) * factor
/tmp/ipykernel_2341/1859564747.py:136: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.  6.  4.5 ... 0.  0.  4.5]' has dtype incompatible with int64, please explicitly cast to a compatible d


========== SUMMARY ==========
dataset  mi_type  n    rca_r1    rca_r3    rca_r5   rca_r10  mean_best_rank  mean_rank_corr
 COMPAS   I(Z;A) 35 88.571429 91.428571 94.285714 97.142857        1.800000        0.674607
 COMPAS I(Z;A|Y) 35 85.714286 85.714286 91.428571 97.142857        2.085714        0.674607

========== SCENARIO SUMMARY ==========
                       scenario  mi_type  n  rca_r1  rca_r3  rca_r5  mean_best_rank  mean_rank_corr
          M2_MISSING_PLUS_SCALE   I(Z;A)  5   100.0   100.0   100.0             1.0        0.734403
          M2_MISSING_PLUS_SCALE I(Z;A|Y)  5   100.0   100.0   100.0             1.0        0.734403
          M3_SCALE_PLUS_MASKING   I(Z;A)  5   100.0   100.0   100.0             1.0        0.675089
          M3_SCALE_PLUS_MASKING I(Z;A|Y)  5   100.0   100.0   100.0             1.0        0.675089
S1_CONDITIONAL_SELECTION_PRIORS   I(Z;A)  5    20.0    40.0    60.0             6.6        0.564024
S1_CONDITIONAL_SELECTION_PRIORS I(Z;A|Y)  5     0.0  